# Modèle Proie-Prédateur Simplifié

Ce notebook démontre l'architecture complète de Seapopym avec un modèle à 2 espèces :
- **Espèce 1 (Proie)** : Croissance constante
- **Espèce 2 (Prédateur)** : Consomme la proie

Objectif : Observer l'évolution des biomasses sur une simulation de 30 jours.

In [ ]:
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.standard.coordinates import Coordinates

print("✅ Imports réussis")

## 1. Définition du Modèle Scientifique

### Équations

**Proie (Population/biomass_prey)** :
$$\frac{dP}{dt} = r - m \cdot \text{Prédateur}$$

Où :
- $r = 0.1$ kg/m²/jour : taux de croissance constant
- $m = 0.05$ kg/m²/jour par kg/m² de prédateur : taux de mortalité par prédation

**Prédateur (Population/biomass_predator)** :
$$\frac{dD}{dt} = e \cdot \text{Proie} - d$$

Où :
- $e = 0.02$ : efficacité de conversion proie → prédateur (sans unité)
- $d = 0.05$ kg/m²/jour : mortalité naturelle du prédateur

**Note sur les unités** :
- Biomasses stockées en g/m² dans le modèle
- Taux convertis en g/m²/s pour l'intégration temporelle
- Affichage en kg/m² pour la lisibilité

**Architecture** :
- Les deux espèces sont dans le même groupe "Population" pour permettre les interactions

In [ ]:
# Fonctions de calcul des tendances


def prey_growth(biomass_prey):
    """Croissance constante de la proie."""
    # Croissance : +0.1 kg/m²/jour = 100 g/m²/jour = 100/86400 g/m²/s
    growth_rate_per_second = 100.0 / 86400
    return {"growth": biomass_prey * 0 + growth_rate_per_second}  # Constante


def predation(biomass_prey, biomass_predator):
    """Mortalité de la proie par prédation."""
    # Mortalité = -0.05 * prédateur (en g/m²/s)
    # 0.05 kg/jour * biomass_predator = 0.05/86400 kg/s * biomass_predator
    # = 50/86400 g/s/m² per kg/m² of predator
    # Since biomass is in g/m², we need: 50/86400 / 1000 * biomass_predator
    mortality_rate_per_second = -0.05 * biomass_predator / 86400
    return {"mortality": mortality_rate_per_second}


def predator_gain(biomass_prey):
    """Gain du prédateur en consommant la proie."""
    # Gain = 0.02 * proie (en g/m²/s)
    # 0.02 kg/jour * biomass_prey = 0.02/86400 kg/s * biomass_prey
    # = 20/86400 g/s/m² per kg/m² of prey
    # Since biomass is in g/m², we need: 20/86400 / 1000 * biomass_prey
    gain_rate_per_second = 0.02 * biomass_prey / 86400
    return {"gain": gain_rate_per_second}


def predator_mortality(biomass_predator):
    """Mortalité naturelle du prédateur."""
    # Mortalité = -0.05 kg/m²/jour = -50 g/m²/jour = -50/86400 g/m²/s
    mortality_rate_per_second = -50.0 / 86400
    return {"mortality": biomass_predator * 0 + mortality_rate_per_second}  # Constante


print("✅ Fonctions de calcul définies")

## 2. Configuration du Blueprint

Le Blueprint va :
1. Enregistrer les biomasses initiales comme forçages
2. Enregistrer les unités de calcul avec leurs tendances
3. Compiler le graphe de dépendances

In [ ]:
def configure_predator_prey_model(bp: Blueprint):
    """Configure le modèle proie-prédateur dans le Blueprint."""

    bp.register_group(
        "Population",
        [
            {
                "func": prey_growth,
                "input_mapping": {"biomass_prey": "biomass_prey"},
                "output_mapping": {"growth": "prey_growth_rate"},
                "output_tendencies": {"growth": "biomass_prey"},
                "output_units": {"growth": "g/m**2/second"},
            },
            {
                "func": predation,
                "input_mapping": {
                    "biomass_prey": "biomass_prey",
                    "biomass_predator": "biomass_predator",
                },
                "output_mapping": {"mortality": "prey_mortality_rate"},
                "output_tendencies": {"mortality": "biomass_prey"},
                "output_units": {"mortality": "g/m**2/second"},
            },
            {
                "func": predator_gain,
                "input_mapping": {"biomass_prey": "biomass_prey"},
                "output_mapping": {"gain": "predator_gain_rate"},
                "output_tendencies": {"gain": "biomass_predator"},
                "output_units": {"gain": "g/m**2/second"},
            },
            {
                "func": predator_mortality,
                "input_mapping": {"biomass_predator": "biomass_predator"},
                "output_mapping": {"mortality": "predator_mortality_rate"},
                "output_tendencies": {"mortality": "biomass_predator"},
                "output_units": {"mortality": "g/m**2/second"},
            },
        ],
        state_variables={
            "biomass_prey": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
            "biomass_predator": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
        },
    )


print("✅ Configuration du modèle définie")

In [ ]:
bp = Blueprint()
configure_predator_prey_model(bp)
plan = bp.build()

fig = (bp.visualize(title="Modèle Proie-Prédateur : Graphe de Dépendances"),)
plt.show()

## 3. État Initial

Grille simplifiée 1x1 avec :
- Biomasse initiale proie : 10 kg/m²
- Biomasse initiale prédateur : 2 kg/m²

In [ ]:
initial_state = xr.Dataset(
    {
        "Population/biomass_prey": (("x"), [10000.0]),  # 10 kg/m² = 10000 g/m²
        "Population/biomass_predator": (("x"), [2000.0]),  # 2 kg/m² = 2000 g/m²
    },
    coords={"x": [0]},
)

print("État initial :")
print(initial_state)
print("\n✅ État initial créé")

## 4. Configuration de la Simulation

Simulation de 30 jours avec un pas de temps de 1 jour.

In [ ]:
config = SimulationConfig(
    start_date=datetime(2000, 1, 1),
    end_date=datetime(2002, 1, 1),  # 30 jours
    timestep=timedelta(days=1),
)

print(f"Configuration : {config.start_date} → {config.end_date}")
print(f"Timestep : {config.timestep}")
print("\n✅ Configuration créée")

## 5. Exécution de la Simulation

In [ ]:
# Création du controller
controller = SimulationController(config)

# Setup avec notre configuration
controller.setup(
    configure_predator_prey_model,
    initial_state,
    output_variables=["Population/biomass_prey", "Population/biomass_predator"],
)

print("Setup terminé.")
print(f"Tendency map : {controller.execution_plan.tendency_map}")
print("\n✅ Setup réussi")

In [ ]:
# Exécution de la simulation
print("Démarrage de la simulation...")
controller.run()
print("Simulation terminée.")

# Extraction des résultats
results = controller.results

# Conversion en listes pour visualisation
results_prey = results["Population/biomass_prey"].values.flatten().tolist()
results_predator = results["Population/biomass_predator"].values.flatten().tolist()
days = list(range(len(results_prey)))

print(f"\n✅ Simulation terminée : {len(days)} jours")

## 6. Visualisation des Résultats

In [ ]:
# Conversion g/m² → kg/m² pour la visualisation
results_prey_kg = [x / 1000 for x in results_prey]
results_predator_kg = [x / 1000 for x in results_predator]

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(days, results_prey_kg, "o-", color="green", label="Proie", linewidth=2)
plt.plot(days, results_predator_kg, "s-", color="red", label="Prédateur", linewidth=2)
plt.xlabel("Jours", fontsize=12)
plt.ylabel("Biomasse (kg/m²)", fontsize=12)
plt.title("Évolution des Biomasses", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(results_prey_kg, results_predator_kg, "o-", color="purple", linewidth=2, markersize=4)
plt.plot(results_prey_kg[0], results_predator_kg[0], "go", markersize=10, label="Départ")
plt.plot(results_prey_kg[-1], results_predator_kg[-1], "rs", markersize=10, label="Arrivée")
plt.xlabel("Biomasse Proie (kg/m²)", fontsize=12)
plt.ylabel("Biomasse Prédateur (kg/m²)", fontsize=12)
plt.title("Trajectoire dans l'Espace des Phases", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Visualisation générée")

## 7. Statistiques Finales

In [ ]:
print("=" * 50)
print("STATISTIQUES FINALES")
print("=" * 50)

# Conversion g/m² → kg/m² pour l'affichage
prey_init_kg = results_prey[0] / 1000
prey_final_kg = results_prey[-1] / 1000
predator_init_kg = results_predator[0] / 1000
predator_final_kg = results_predator[-1] / 1000

print(f"\nBiomasse Proie :")
print(f"  - Initiale : {prey_init_kg:.2f} kg/m²")
print(f"  - Finale   : {prey_final_kg:.2f} kg/m²")
print(f"  - Variation : {prey_final_kg - prey_init_kg:+.2f} kg/m²")

print(f"\nBiomasse Prédateur :")
print(f"  - Initiale : {predator_init_kg:.2f} kg/m²")
print(f"  - Finale   : {predator_final_kg:.2f} kg/m²")
print(f"  - Variation : {predator_final_kg - predator_init_kg:+.2f} kg/m²")

print(f"\nBiomasse Totale :")
total_init = prey_init_kg + predator_init_kg
total_final = prey_final_kg + predator_final_kg
print(f"  - Initiale : {total_init:.2f} kg/m²")
print(f"  - Finale   : {total_final:.2f} kg/m²")
print(f"  - Variation : {total_final - total_init:+.2f} kg/m²")
print("\n" + "=" * 50)

## 8. Inspection de l'Architecture

Vérifions comment le Blueprint a organisé le modèle.

In [ ]:
print("PLAN D'EXÉCUTION")
print("=" * 50)

for i, (group_name, tasks) in enumerate(controller.execution_plan.task_groups):
    print(f"\n{i + 1}. Groupe : {group_name}")
    for task in tasks:
        print(f"   - {task.name}")
        print(f"     Entrées  : {task.input_mapping}")
        print(f"     Sorties  : {task.output_mapping}")

print("\n" + "=" * 50)
print("\nTENDANCES CONFIGURÉES")
print("=" * 50)
for var, tendencies in controller.execution_plan.tendency_map.items():
    print(f"\n{var} ← {tendencies}")

print("\n✅ Inspection terminée")